[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajitpanday80/ai-learn/blob/main/notebooks/llm-need.ipynb)

# The Need for LLMs — Hands-On Lab

In this notebook we'll rebuild the *history of NLP* in miniature, on CPU only (no GPU needed):

1. A **rule-based chatbot** — and watch it break on paraphrases.
2. A **statistical n-gram model** — and watch it lose coherence.
3. A **small pretrained LLM** (DistilGPT-2) — and see fluent, context-aware generation.
4. A chart showing **why scale matters**.

Run each cell in order.

In [ ]:
!pip install -q transformers torch matplotlib

## 1. Rule-Based Chatbot

This is how a 1980s-style chatbot worked: match keywords, return a canned response. Try it with normal phrasing, then with a paraphrase — watch it fail.

In [ ]:
def rule_based_bot(text):
    text = text.lower()
    if 'hello' in text or 'hi' in text:
        return "Hello! How can I help you?"
    elif 'weather' in text:
        return "I don't have weather data."
    elif 'bye' in text:
        return "Goodbye!"
    else:
        return "I don't understand."

test_inputs = [
    'hello there',
    'hi, good morning',
    'yo whats up',          # paraphrase of a greeting -> will fail
    "what's the weather like today",
    'is it going to rain',  # paraphrase of weather question -> will fail
]

for t in test_inputs:
    print(f"USER: {t}\nBOT:  {rule_based_bot(t)}\n")

## 2. Statistical N-gram Model

Now we build a tiny bigram (Markov chain) language model from a small corpus and let it *generate* text by sampling. Notice how it stays locally plausible (word-to-word) but drifts into incoherence over longer stretches.

In [ ]:
import random
from collections import defaultdict

corpus = """
the cat sat on the mat. the dog sat on the rug.
the cat chased the dog. the dog chased the cat.
the mat was soft. the rug was old.
"""

words = corpus.lower().replace('.', ' .').split()

bigram_counts = defaultdict(list)
for w1, w2 in zip(words[:-1], words[1:]):
    bigram_counts[w1].append(w2)

def generate_ngram_text(start_word, n_words=20, seed=None):
    if seed is not None:
        random.seed(seed)
    current = start_word
    output = [current]
    for _ in range(n_words - 1):
        choices = bigram_counts.get(current)
        if not choices:
            break
        current = random.choice(choices)
        output.append(current)
    return ' '.join(output)

print(generate_ngram_text('the', n_words=25, seed=1))
print(generate_ngram_text('the', n_words=25, seed=7))

**Experiment:** change the `seed` value above and re-run. Notice the output is grammatically local (bigrams are valid pairs) but has no real long-range meaning or memory of what was said earlier — this is exactly the wall statistical NLP hit.

## 3. A Small LLM (DistilGPT-2)

DistilGPT-2 is a small (~82M parameter) Transformer language model — tiny by modern standards, but it already shows the qualitative jump over rules and n-grams: it uses long-range context and produces coherent continuations. This runs comfortably on CPU.

In [ ]:
from transformers import pipeline, set_seed

generator = pipeline('text-generation', model='distilgpt2', device=-1)  # device=-1 -> CPU
set_seed(42)

prompt = "The cat chased the dog because"
result = generator(prompt, max_new_tokens=30, num_return_sequences=1, do_sample=True, temperature=0.8)
print(result[0]['generated_text'])

In [ ]:
# Compare all three approaches side-by-side on the same theme
prompts = ["yo whats up", "the cat chased the dog because"]

print("Rule-based bot:")
print(' ', rule_based_bot(prompts[0]))

print("\nN-gram model (no real concept of 'because'):")
print(' ', generate_ngram_text('the', n_words=15, seed=3))

print("\nSmall LLM (distilgpt2):")
out = generator(prompts[1], max_new_tokens=25, do_sample=True, temperature=0.8)
print(' ', out[0]['generated_text'])

## 4. Why Scale Matters

The jump in quality from n-grams to Transformers is only half the story — the other half is *scale*. Let's plot how model sizes grew over time (approximate parameter counts, log scale).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ['GPT-1\n(2018)', 'GPT-2\n(2019)', 'GPT-3\n(2020)', 'PaLM\n(2022)', 'GPT-4-class\n(2023+)']
params_millions = [117, 1500, 175000, 540000, 1000000]  # approximate, illustrative

fig, ax = plt.subplots(figsize=(9, 5.5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

colors = ['#00d4ff', '#7c4dff', '#ff4081', '#ffca28', '#69f0ae']
bars = ax.bar(models, params_millions, color=colors)

ax.set_yscale('log')
ax.set_ylabel('Parameters (millions, log scale)', color='white')
ax.set_title('Model Scale Growth Over Time (illustrative)', color='white', fontsize=14)

ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('white')

for bar, val in zip(bars, params_millions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.15, f'{val:,}M',
            ha='center', color='white', fontsize=9)

plt.tight_layout()
plt.show()

## Experiments to Try

- Change the `prompt` in Section 3 to something requiring context (e.g. `"She opened the fridge and found"`) and see how the LLM stays on-topic.
- Add more sentences to the `corpus` in Section 2 and see if the n-gram output improves — does it ever match LLM coherence?
- Try `temperature=0.2` vs `temperature=1.2` in the generator call — lower is more deterministic, higher is more random.

## Takeaway

Rule-based systems need exhaustive manual coverage. N-gram models forget context after a few words. Even a *small* Transformer LLM like DistilGPT-2 already generalizes better — and that gap only widens with scale, which is exactly why the field moved toward LLMs.